# Feature Engineering and Final Kaggle Submission

My current models — XGBoost (84.92% validation accuracy, Kaggle: 0.77751) and SVM (82.12%, Kaggle: 0.77751) — scored identically on unseen test data despite meaningfully different validation accuracies. This tells me the models have already extracted everything they can from the current features. The bottleneck is not the algorithm — it’s the information available to it.

Feature engineering is the process of constructing more informative inputs from raw data. Rather than giving a model a passenger’s raw Name string (useless) or SibSp and Parch separately, we extract what those columns actually encode: social status, age group, whether someone was travelling alone. We are translating domain knowledge into signal the model can use.

In this notebook, we will use XGBoost's model as a baseline and add five feature upgrades sequentially — submitting to Kaggle after each — to measure the isolated impact of each addition. The hypothesis: better features will close the gap between validation performance and generalisation, and will lift XGBoost's model beyond its current ceiling of 0.77751.

## 01. Baseline

Original XGBoost model using n_estimators=10, max_depth=7, learning_rate=0.3.

Reference:
Validation accuracy - 0.8492
Kaggle score - 0.77751

In [2]:
# Import Modules
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
print("Set Up Complete")

# Load Data
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

Set Up Complete


In [4]:
# Save test passenger IDs for submission
test_passenger_id = test["PassengerId"] 

# Drop columns not used in baseline
train_dropped = train.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])
test_dropped = test.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])

# Separate feature and target, then Train/Test Split
X = train_dropped.drop(columns=["Survived"])
y = train_dropped["Survived"]
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

# Separate numerical and categorical columns
numerical_columns = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
categorical_columns = ["Sex", "Embarked"]

# Build preprocessing
numerical_transformer = Pipeline([("imputer", SimpleImputer(strategy="median"))])
categorical_transformer = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                                    ("encoder", OneHotEncoder())])

# Combine both
preprocessing = ColumnTransformer([("num", numerical_transformer, numerical_columns),
                                   ("cat", categorical_transformer, categorical_columns)],
                                   remainder = 'passthrough')

# Train baseline XGBoost model with same hyperparameters as before
xgb = Pipeline([("preprocessing", preprocessing), 
                ("classifier", XGBClassifier(
                    n_estimators=10, 
                    max_depth=7, 
                    learning_rate=0.3, 
                    random_state=42, 
                    eval_metric="logloss"))])
xgb.fit(X_train, y_train)

# Check validation accuracy
xgb_valid_pred = xgb.predict(X_valid)
xgb_valid_accuracy = accuracy_score(y_valid, xgb_valid_pred)
print(f"Baseline validation accuracy: {xgb_valid_accuracy:.4f}")

Baseline validation accuracy: 0.8492


In [5]:
# Baseline submission
xgb_test_pred = xgb.predict(test_dropped)
baseline_submission = pd.DataFrame({"PassengerId": test_passenger_id, "Survived": xgb_test_pred})
baseline_submission.to_csv("titanic_submission_v5.1_baseline.csv", index=False)
print(baseline_submission.shape)

(418, 2)


## 02. Feature Eng 1: Title from Name

Extract titles from Name: Mr, Mrs, Miss, Master, Rare. 
- Rare includes titles like "Don", "Rev", "Capt"...
- Master are for titles used for children below age 12. 

Titles can encode useful demographic information such as gender, age, and social status, that isn't fully represented by the existing features.

In [13]:
# Data checking for title feature

# Check all unique titles in the dataset
train["Title"] = train["Name"].str.extract(r",\s*([^.]*)\.", expand=False)
print(train['Title'].unique())

# Filter for "Master" titles and get their age range
master_ages = train[train['Title'] == 'Master']['Age']
print(f"Age range for 'Master' titles: {master_ages.min()} - {master_ages.max()}")

<StringArray>
[          'Mr',          'Mrs',         'Miss',       'Master',
          'Don',          'Rev',           'Dr',          'Mme',
           'Ms',        'Major',         'Lady',          'Sir',
         'Mlle',          'Col',         'Capt', 'the Countess',
     'Jonkheer']
Length: 17, dtype: str
Age range for 'Master' titles: 0.42 - 12.0


In [ ]:
# Create add_title() function to extract title from name
def add_title(df):
    df = df.copy()
    # Extract title from name
    df["Title"] = df["Name"].str.extract(r",\s*([^.]*)\.", expand=False)
    # Group uncommon titles into Rare
    df["Title"] = df["Title"].replace(["Lady", "Countess","Capt", "Col",
        "Don", "Dr", "Major", "Rev", "Sir", "Jonkheer", "Dona", "the Countess"], "Rare")
    # Standardise equivalent titles
    df["Title"] = df["Title"].replace(["Mlle", "Ms"], "Miss")
    df["Title"] = df["Title"].replace("Mme", "Mrs")
    return df

# Apply function to both train and test datasets
train_v2 = add_title(train)
test_v2 = add_title(test)
print(train_v2["Title"].value_counts())

Title
Mr        517
Miss      185
Mrs       126
Master     40
Rare       23
Name: count, dtype: int64


In [11]:
# Drop columns not used while keeping the new Title feature
train_v2_dropped = train_v2.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])
test_v2_dropped = test_v2.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])

# Separate feature and target
X_v2 = train_v2_dropped.drop(columns=["Survived"])
y_v2 = train_v2_dropped["Survived"]

# Train/Test Split
X_train_v2, X_valid_v2, y_train_v2, y_valid_v2 = train_test_split(X_v2, y_v2, test_size=0.2, random_state=42)

# Update preprocessing to include the new Title feature
numerical_columns_v2 = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
categorical_columns_v2 = ["Sex", "Embarked", "Title"]

# Build preprocessing
numerical_transformer = Pipeline([("imputer", SimpleImputer(strategy="median"))])
categorical_transformer = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                                    ("encoder", OneHotEncoder())])

# Combine both
preprocessing = ColumnTransformer([("num", numerical_transformer, numerical_columns_v2),
                                   ("cat", categorical_transformer, categorical_columns_v2)],
                                   remainder = 'passthrough')

# Train baseline XGBoost model with same hyperparameters as before
xgb_v2 = Pipeline([("preprocessing", preprocessing), 
                ("classifier", XGBClassifier(
                    n_estimators=10, 
                    max_depth=7, 
                    learning_rate=0.3, 
                    random_state=42, 
                    eval_metric="logloss"))])
xgb_v2.fit(X_train_v2, y_train_v2)

# Check validation accuracy
xgb_v2_valid_pred = xgb_v2.predict(X_valid_v2)
xgb_v2_valid_accuracy = accuracy_score(y_valid_v2, xgb_v2_valid_pred)
print(f"V2 validation accuracy: {xgb_v2_valid_accuracy:.4f}")

# V2 submission
xgb_v2_test_pred = xgb_v2.predict(test_v2_dropped)
v2_submission = pd.DataFrame({"PassengerId": test_passenger_id, "Survived": xgb_v2_test_pred})
v2_submission.to_csv("titanic_submission_v5.2_title_feature.csv", index=False)
print(v2_submission.shape)

V2 validation accuracy: 0.8603
(418, 2)


V2: added title feature
Validation accuracy: 0.8603
Kaggle score: 0.75358

The validation accuracy increased from 0.8492 to 0.8603, but Kaggle score decreased from 0.77751 to 0.75358.

This suggests the Title feature helps the model on this validation split, but that improvement doesn't generalise to Kaggle's test set. It could reflect sampling variability or overfitting, though it can't conclude which from these two scores alone.

## 03. Feature Eng 2: Family features

Add FamilySize = SibSp + Parch + 1.
- Family Size: total number of family members on board, including passenger (+1)

These capture group sizes. Travelling with family might affect survival beyond what SibSp and Parch may tell us individually.

In [26]:
# Create add_family_size() function to create family size feature
def add_family_size(df):
    df = df.copy()
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    return df

# Apply function to both train and test datasets
train_v3 = add_family_size(pd.read_csv("train.csv"))
test_v3 = add_family_size(pd.read_csv("test.csv"))
print(train_v3["FamilySize"].value_counts())

FamilySize
1     537
2     161
3     102
4      29
6      22
5      15
7      12
11      7
8       6
Name: count, dtype: int64


In [34]:
# Drop columns not used while keeping the new family size feature
train_v3_dropped = train_v3.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])
test_v3_dropped = test_v3.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])

# Separate feature and target
X_v3 = train_v3_dropped.drop(columns=["Survived"])
y_v3 = train_v3_dropped["Survived"]

# Train/Test Split
X_train_v3, X_valid_v3, y_train_v3, y_valid_v3 = train_test_split(X_v3, y_v3, test_size=0.2, random_state=42)

# Update preprocessing to include the new FamilySize feature
print(X_train_v3.columns.tolist())
numerical_columns_v3 = ["Pclass", "Age", "SibSp", "Parch", "Fare", "FamilySize"]
categorical_columns_v3 = ["Sex", "Embarked"]

# Build preprocessing
numerical_transformer = Pipeline([("imputer", SimpleImputer(strategy="median"))])
categorical_transformer = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                                    ("encoder", OneHotEncoder())])

# Combine both
preprocessing = ColumnTransformer([("num", numerical_transformer, numerical_columns_v3),
                                   ("cat", categorical_transformer, categorical_columns_v3)],
                                   remainder = 'passthrough')

# Train baseline XGBoost model with same hyperparameters as before
xgb_v3 = Pipeline([("preprocessing", preprocessing), 
                ("classifier", XGBClassifier(
                    n_estimators=10, 
                    max_depth=7, 
                    learning_rate=0.3, 
                    random_state=42, 
                    eval_metric="logloss"))])
xgb_v3.fit(X_train_v3, y_train_v3)

# Check validation accuracy
xgb_v3_valid_pred = xgb_v3.predict(X_valid_v3)
xgb_v3_valid_accuracy = accuracy_score(y_valid_v3, xgb_v3_valid_pred)
print(f"V3 validation accuracy: {xgb_v3_valid_accuracy:.4f}")

# V3 submission
xgb_v3_test_pred = xgb_v3.predict(test_v3_dropped)
v3_submission = pd.DataFrame({"PassengerId": test_passenger_id, "Survived": xgb_v3_test_pred})
v3_submission.to_csv("titanic_submission_v5.3_family_size_feature.csv", index=False)
print(v3_submission.shape)

['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'FamilySize']
V3 validation accuracy: 0.8324
(418, 2)


V3: added family size feature
Validation accuracy: 0.8324
Kaggle score: 0.77990

The validation accuracy decreased from 0.8492 to 0.8324, but Kaggle score increased from 0.77751 to 0.77990.

This suggests FamilySize may help predictors on the unseen Kaggle test set.

## 04. Feature Eng 3: IsAlone feature

Add IsAlone feature
- IsAlone: Binary classification -- 1 if alone, else 0

In [30]:
# Create add_is_alone() function to create is alone feature (0 = not alone, 1 = alone)
def add_is_alone(df):
    df = df.copy()
    df["IsAlone"] = ((df["SibSp"] == 0) & (df["Parch"] == 0)).astype(int)   
    return df

# Apply function to both train and test datasets
train_v4 = add_is_alone(pd.read_csv("train.csv"))
test_v4 = add_is_alone(pd.read_csv("test.csv"))
print(train_v4["IsAlone"].value_counts())

IsAlone
1    537
0    354
Name: count, dtype: int64


In [35]:
# Drop columns not used while keeping the new IsAlone feature
train_v4_dropped = train_v4.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])
test_v4_dropped = test_v4.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])

# Separate feature and target
X_v4 = train_v4_dropped.drop(columns=["Survived"])
y_v4 = train_v4_dropped["Survived"]

# Train/Test Split
X_train_v4, X_valid_v4, y_train_v4, y_valid_v4 = train_test_split(X_v4, y_v4, test_size=0.2, random_state=42)

# Update preprocessing to include the new IsAlone feature under categorical columns
print(X_train_v4.columns.tolist())
numerical_columns_v4 = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
categorical_columns_v4 = ["Sex", "Embarked", "IsAlone"]

# Build preprocessing
numerical_transformer = Pipeline([("imputer", SimpleImputer(strategy="median"))])
categorical_transformer = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                                    ("encoder", OneHotEncoder())])

# Combine both
preprocessing = ColumnTransformer([("num", numerical_transformer, numerical_columns_v4),
                                   ("cat", categorical_transformer, categorical_columns_v4)],
                                   remainder = 'passthrough')

# Train baseline XGBoost model with same hyperparameters as before
xgb_v4 = Pipeline([("preprocessing", preprocessing), 
                ("classifier", XGBClassifier(
                    n_estimators=10, 
                    max_depth=7, 
                    learning_rate=0.3, 
                    random_state=42, 
                    eval_metric="logloss"))])
xgb_v4.fit(X_train_v4, y_train_v4)

# Check validation accuracy
xgb_v4_valid_pred = xgb_v4.predict(X_valid_v4)
xgb_v4_valid_accuracy = accuracy_score(y_valid_v4, xgb_v4_valid_pred)
print(f"V4 validation accuracy: {xgb_v4_valid_accuracy:.4f}")

# V4 submission
xgb_v4_test_pred = xgb_v4.predict(test_v4_dropped)
v4_submission = pd.DataFrame({"PassengerId": test_passenger_id, "Survived": xgb_v4_test_pred})
v4_submission.to_csv("titanic_submission_v5.4_is_alone_feature.csv", index=False)
print(v4_submission.shape)

['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'IsAlone']
V4 validation accuracy: 0.8492
(418, 2)


V4: added isalone feature
Validation accuracy: 0.8492
Kaggle score: 0.77751

Both scores matched the baseline exactly, suggesting that adding this feature alone did not change predictions on either set. Since IsAlone is derived from SibSp and Parch, the baseline model may already have captured relevant information from those features.

## 05. Feature Eng 4: Age groups

Instead of using Age as a continuous variable, create child, teen, adult, senior categories using specified age boundaries.
Child: 12 and below
Teen: 13 to 17
Adult: 18 to 59
Senior: 60 and above
Missing ages (177 out of 891 missing) remain missing and will be handled by the categorical imputer.

In [36]:
print(train.Age.describe())

count    714.000000
mean      29.699118
std       14.526497
min        0.420000
25%       20.125000
50%       28.000000
75%       38.000000
max       80.000000
Name: Age, dtype: float64


In [37]:
# Create add_age_group() function to create age group feature (age range: 0.42 to 80)
def add_age_group(df):
    df = df.copy()
    df["AgeGroup"] = pd.cut(df["Age"], bins=[0, 12, 17, 59, 100], labels=["Child", "Teen", "Adult", "Senior"])
    return df

# Apply function to both train and test datasets
train_v5 = add_age_group(pd.read_csv("train.csv"))
test_v5 = add_age_group(pd.read_csv("test.csv"))
print(train_v5["AgeGroup"].value_counts())

AgeGroup
Adult     575
Child      69
Teen       44
Senior     26
Name: count, dtype: int64


In [38]:
# Drop columns not used while keeping the new AgeGroup feature
train_v5_dropped = train_v5.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])
test_v5_dropped = test_v5.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])

# Separate feature and target
X_v5 = train_v5_dropped.drop(columns=["Survived"])
y_v5 = train_v5_dropped["Survived"]

# Train/Test Split
X_train_v5, X_valid_v5, y_train_v5, y_valid_v5 = train_test_split(X_v5, y_v5, test_size=0.2, random_state=42)

# Update preprocessing to include the new AgeGroup feature under categorical columns
print(X_train_v5.columns.tolist())
numerical_columns_v5 = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
categorical_columns_v5 = ["Sex", "Embarked", "AgeGroup"]

# Build preprocessing
numerical_transformer = Pipeline([("imputer", SimpleImputer(strategy="median"))])
categorical_transformer = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                                    ("encoder", OneHotEncoder())])

# Combine both
preprocessing = ColumnTransformer([("num", numerical_transformer, numerical_columns_v5),
                                   ("cat", categorical_transformer, categorical_columns_v5)],
                                   remainder = 'passthrough')

# Train baseline XGBoost model with same hyperparameters as before
xgb_v5 = Pipeline([("preprocessing", preprocessing), 
                ("classifier", XGBClassifier(
                    n_estimators=10, 
                    max_depth=7, 
                    learning_rate=0.3, 
                    random_state=42, 
                    eval_metric="logloss"))])
xgb_v5.fit(X_train_v5, y_train_v5)

# Check validation accuracy
xgb_v5_valid_pred = xgb_v5.predict(X_valid_v5)
xgb_v5_valid_accuracy = accuracy_score(y_valid_v5, xgb_v5_valid_pred)
print(f"V5 validation accuracy: {xgb_v5_valid_accuracy:.4f}")

# V5 submission
xgb_v5_test_pred = xgb_v5.predict(test_v5_dropped)
v5_submission = pd.DataFrame({"PassengerId": test_passenger_id, "Survived": xgb_v5_test_pred})
v5_submission.to_csv("titanic_submission_v5.5_age_group_feature.csv", index=False)
print(v5_submission.shape)

['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'AgeGroup']
V5 validation accuracy: 0.8492
(418, 2)


V5: added age group feature
Validation accuracy: 0.8492
Kaggle score: 0.78229

Validation accuracy stayed the same, but the Kaggle score increased from 0.77751 to 0.78229. Grouping ages may help the model make better predictions on the unseen Kaggle test set, even though it did not improve accuracy on this validation split. 

## 06. Feature Eng 5: Cabin deck

Extract deck A-G and assign unknown to the 687 NaN values

In [39]:
print(train.Cabin.value_counts(dropna=False))

Cabin
NaN            687
G6               4
C23 C25 C27      4
B96 B98          4
F33              3
              ... 
E17              1
A24              1
C50              1
B42              1
C148             1
Name: count, Length: 148, dtype: int64


In [40]:
# Create add_deck() function to extract deck from cabin (A to G)
def add_deck(df):
    df = df.copy()
    df["Deck"] = df["Cabin"].str[0].fillna("Unknown")
    return df

# Apply function to both train and test datasets
train_v6 = add_deck(pd.read_csv("train.csv"))
test_v6 = add_deck(pd.read_csv("test.csv"))
print(train_v6["Deck"].value_counts())

Deck
Unknown    687
C           59
B           47
D           33
E           32
A           15
F           13
G            4
T            1
Name: count, dtype: int64


In [41]:
# Drop columns not used while keeping the new Deck feature
train_v6_dropped = train_v6.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])
test_v6_dropped = test_v6.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])

# Separate feature and target
X_v6 = train_v6_dropped.drop(columns=["Survived"])
y_v6 = train_v6_dropped["Survived"]

# Train/Test Split
X_train_v6, X_valid_v6, y_train_v6, y_valid_v6 = train_test_split(X_v6, y_v6, test_size=0.2, random_state=42)

# Update preprocessing to include the new Deck feature under categorical columns
print(X_train_v6.columns.tolist())
numerical_columns_v6 = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
categorical_columns_v6 = ["Sex", "Embarked", "Deck"]

# Build preprocessing
numerical_transformer = Pipeline([("imputer", SimpleImputer(strategy="median"))])
categorical_transformer = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                                    ("encoder", OneHotEncoder())])

# Combine both
preprocessing = ColumnTransformer([("num", numerical_transformer, numerical_columns_v6),
                                   ("cat", categorical_transformer, categorical_columns_v6)],
                                   remainder = 'passthrough')

# Train baseline XGBoost model with same hyperparameters as before
xgb_v6 = Pipeline([("preprocessing", preprocessing), 
                ("classifier", XGBClassifier(
                    n_estimators=10, 
                    max_depth=7, 
                    learning_rate=0.3, 
                    random_state=42, 
                    eval_metric="logloss"))])
xgb_v6.fit(X_train_v6, y_train_v6)

# Check validation accuracy
xgb_v6_valid_pred = xgb_v6.predict(X_valid_v6)
xgb_v6_valid_accuracy = accuracy_score(y_valid_v6, xgb_v6_valid_pred)
print(f"V6 validation accuracy: {xgb_v6_valid_accuracy:.4f}")

# V6 submission
xgb_v6_test_pred = xgb_v6.predict(test_v6_dropped)
v6_submission = pd.DataFrame({"PassengerId": test_passenger_id, "Survived": xgb_v6_test_pred})
v6_submission.to_csv("titanic_submission_v6_deck_feature.csv", index=False)
print(v6_submission.shape)

['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'Deck']
V6 validation accuracy: 0.8212
(418, 2)


V6: added deck feature
Validation accuracy: 0.8212
Kaggle score: 0.76555

The Deck feature performed worse than the baseline on both metrics. It may still contain useful information, but its value could be limited by the large number of passengers with missing cabin information, the model settings, or how this feature interacts with other features. 

## 07. Feature Eng 6: All combined

Age group feature produced the best Kaggle score on 0.78229. Family size produced the next best score of 0.77990. Is Alone feature had no observed change. Title feature gave an improved validation accuracy but poorer kaggle score. Deck feature produced worse results than baseline. 

Then combine all feature engineering upgrades and submit a final Kaggle submission. 

| Version | Description | Validation Accuracy | Kaggle score |
| ----- | ------ | ------------------- | -----------: |
| 5.1 |Baseline | 0.8492 | 0.77751 |
| 5.2 | Title feature | 0.8603 | 0.75358 |
| 5.3 | Family size feature | 0.8324 | 0.77990|
| 5.4 | IsAlone feature | 0.8492 | 0.77751 |
| 5.5 | Age Group feature | 0.8492 | 0.78229 |
| 5.6 | Cabin Deck feature | 0.8212 | 0.76555 |
| 5.7 | All combined | ? | ? |